In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
project_folder = Path('/content/drive/MyDrive/Bellabeat Case Study')

data_folder = (
    project_folder
    / '01 Original Data'
    / 'mturkfitbit_export_4.12.16-5.12.16'
    / 'Fitabase Data 4.12.16-5.12.16'
)

print(data_folder)

In [ ]:
csv_files = sorted(data_folder.glob('*.csv'))

print(f'Number of CSV files found: {len(csv_files)}')

for file in csv_files:
    print(file.name)

In [ ]:
daily_activity_file = data_folder / "dailyActivity_merged.csv"
sleep_file = data_folder / "sleepDay_merged.csv"

daily_activity = pd.read_csv(daily_activity_file)
sleep = pd.read_csv(sleep_file)

print("Daily activity rows and columns:", daily_activity.shape)
print("Sleep rows and columns:", sleep.shape)

In [ ]:
print("Daily activity columns:")
print(daily_activity.columns.tolist())

print("\nDaily activity data types:")
print(daily_activity.dtypes)

print("\nSleep columns:")
print(sleep.columns.tolist())

print("\nSleep data types:")
print(sleep.dtypes)

In [ ]:
daily_activity["ActivityDate"] = pd.to_datetime(
    daily_activity["ActivityDate"],
    errors="coerce"
)

sleep["SleepDay"] = pd.to_datetime(
    sleep["SleepDay"],
    errors="coerce"
).dt.date

daily_activity["ActivityDate"] = daily_activity["ActivityDate"].dt.date

print("Invalid ActivityDate values:", daily_activity["ActivityDate"].isna().sum())
print("Invalid SleepDay values:", sleep["SleepDay"].isna().sum())

print("\nDaily activity date range:")
print(daily_activity["ActivityDate"].min(), "to", daily_activity["ActivityDate"].max())

print("\nSleep date range:")
print(sleep["SleepDay"].min(), "to", sleep["SleepDay"].max())

In [ ]:
daily_duplicates = daily_activity.duplicated().sum()
sleep_duplicates = sleep.duplicated().sum()

print("Duplicate daily activity rows:", daily_duplicates)
print("Duplicate sleep rows:", sleep_duplicates)

In [ ]:
rows_before = len(sleep)

sleep = sleep.drop_duplicates().copy()

rows_after = len(sleep)

print("Sleep rows before:", rows_before)
print("Sleep rows after:", rows_after)
print("Duplicate sleep rows removed:", rows_before - rows_after)

In [ ]:
print("Missing values in daily activity:")
print(daily_activity.isna().sum())

print("\nMissing values in sleep:")
print(sleep.isna().sum())

In [ ]:
print("Unique users in daily activity:", daily_activity["Id"].nunique())
print("Unique users in sleep:", sleep["Id"].nunique())

In [ ]:
daily_activity["DayOfWeek"] = pd.to_datetime(
    daily_activity["ActivityDate"]
).dt.day_name()

daily_activity["TotalActiveMinutes"] = (
    daily_activity["VeryActiveMinutes"]
    + daily_activity["FairlyActiveMinutes"]
    + daily_activity["LightlyActiveMinutes"]
)

sleep["SleepHours"] = sleep["TotalMinutesAsleep"] / 60
sleep["TimeInBedHours"] = sleep["TotalTimeInBed"] / 60
sleep["SleepEfficiency"] = (
    sleep["TotalMinutesAsleep"] / sleep["TotalTimeInBed"] * 100
).round(2)

print(daily_activity[
    ["ActivityDate", "DayOfWeek", "TotalActiveMinutes"]
].head())

print()

print(sleep[
    ["SleepDay", "SleepHours", "TimeInBedHours", "SleepEfficiency"]
].head())

In [ ]:
activity_summary = daily_activity[
    [
        "TotalSteps",
        "TotalDistance",
        "VeryActiveMinutes",
        "FairlyActiveMinutes",
        "LightlyActiveMinutes",
        "SedentaryMinutes",
        "TotalActiveMinutes",
        "Calories"
    ]
].describe().round(2)

sleep_summary = sleep[
    [
        "TotalMinutesAsleep",
        "TotalTimeInBed",
        "SleepHours",
        "SleepEfficiency"
    ]
].describe().round(2)

print("Activity summary:")
print(activity_summary)

print("\nSleep summary:")
print(sleep_summary)

In [ ]:
steps_calories_correlation = daily_activity[
    ["TotalSteps", "Calories"]
].corr().iloc[0, 1]

print(
    "Correlation between total steps and calories:",
    round(steps_calories_correlation, 2)
)

In [ ]:
activity_sleep = pd.merge(
    daily_activity,
    sleep,
    left_on=["Id", "ActivityDate"],
    right_on=["Id", "SleepDay"],
    how="inner"
)

print("Combined activity and sleep rows:", len(activity_sleep))
print("Unique users in combined data:", activity_sleep["Id"].nunique())

In [ ]:
sleep_correlations = activity_sleep[
    [
        "TotalSteps",
        "TotalActiveMinutes",
        "SedentaryMinutes",
        "SleepHours"
    ]
].corr()["SleepHours"].round(2)

print("Correlations with sleep hours:")
print(sleep_correlations)

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

activity_by_day = (
    daily_activity.groupby("DayOfWeek")
    .agg(
        average_steps=("TotalSteps", "mean"),
        average_active_minutes=("TotalActiveMinutes", "mean"),
        average_sedentary_minutes=("SedentaryMinutes", "mean"),
        average_calories=("Calories", "mean")
    )
    .reindex(day_order)
    .round(2)
)

print(activity_by_day)

In [ ]:
sleep["DayOfWeek"] = pd.to_datetime(
    sleep["SleepDay"]
).dt.day_name()

sleep_by_day = (
    sleep.groupby("DayOfWeek")
    .agg(
        average_sleep_hours=("SleepHours", "mean"),
        average_time_in_bed=("TimeInBedHours", "mean"),
        average_sleep_efficiency=("SleepEfficiency", "mean")
    )
    .reindex(day_order)
    .round(2)
)

print(sleep_by_day)

In [ ]:
user_activity = (
    daily_activity.groupby("Id")
    .agg(
        average_daily_steps=("TotalSteps", "mean"),
        average_active_minutes=("TotalActiveMinutes", "mean"),
        average_sedentary_minutes=("SedentaryMinutes", "mean")
    )
    .reset_index()
)

def classify_activity(steps):
    if steps < 5000:
        return "Sedentary"
    elif steps < 7500:
        return "Lightly Active"
    elif steps < 10000:
        return "Fairly Active"
    else:
        return "Very Active"

user_activity["ActivityLevel"] = (
    user_activity["average_daily_steps"].apply(classify_activity)
)

print(user_activity["ActivityLevel"].value_counts())

In [ ]:
activity_level_summary = (
    user_activity["ActivityLevel"]
    .value_counts()
    .rename_axis("ActivityLevel")
    .reset_index(name="NumberOfUsers")
)

activity_level_summary["Percentage"] = (
    activity_level_summary["NumberOfUsers"]
    / activity_level_summary["NumberOfUsers"].sum()
    * 100
).round(2)

print(activity_level_summary)

In [ ]:
cleaned_data_folder = project_folder / "02 Cleaned Data"
cleaned_data_folder.mkdir(parents=True, exist_ok=True)

daily_activity.to_csv(
    cleaned_data_folder / "bellabeat_daily_activity_cleaned.csv",
    index=False
)

sleep.to_csv(
    cleaned_data_folder / "bellabeat_sleep_cleaned.csv",
    index=False
)

activity_sleep.to_csv(
    cleaned_data_folder / "bellabeat_activity_sleep_merged.csv",
    index=False
)

print("Cleaned files saved successfully.")
print("Daily activity rows:", len(daily_activity))
print("Sleep rows:", len(sleep))
print("Merged rows:", len(activity_sleep))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [ ]:
chart_folder = project_folder / "04 Visualisations"
chart_folder.mkdir(parents=True, exist_ok=True)

print(chart_folder)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    activity_by_day.index,
    activity_by_day["average_steps"]
)

plt.title("Average Daily Steps by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Average Steps")
plt.xticks(rotation=45)

plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

plt.tight_layout()

chart_path = chart_folder / "average_steps_by_day.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    sleep_by_day.index,
    sleep_by_day["average_sleep_hours"]
)

plt.title("Average Sleep Hours by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Average Sleep Hours")
plt.xticks(rotation=45)

plt.tight_layout()

chart_path = chart_folder / "average_sleep_hours_by_day.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
activity_level_order = [
    "Sedentary",
    "Lightly Active",
    "Fairly Active",
    "Very Active"
]

activity_level_chart = (
    activity_level_summary
    .set_index("ActivityLevel")
    .reindex(activity_level_order)
)

plt.figure(figsize=(8, 5))

plt.bar(
    activity_level_chart.index,
    activity_level_chart["Percentage"]
)

plt.title("Distribution of User Activity Levels")
plt.xlabel("Activity Level")
plt.ylabel("Percentage of Users")
plt.xticks(rotation=20)

plt.tight_layout()

chart_path = chart_folder / "user_activity_levels.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    daily_activity["TotalSteps"],
    daily_activity["Calories"],
    alpha=0.5
)

plt.title("Relationship Between Daily Steps and Calories Burned")
plt.xlabel("Total Daily Steps")
plt.ylabel("Calories Burned")

plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

plt.tight_layout()

chart_path = chart_folder / "steps_vs_calories.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    activity_sleep["SedentaryMinutes"],
    activity_sleep["SleepHours"],
    alpha=0.5
)

plt.title("Relationship Between Sedentary Time and Sleep Duration")
plt.xlabel("Sedentary Minutes")
plt.ylabel("Sleep Hours")

plt.tight_layout()

chart_path = chart_folder / "sedentary_time_vs_sleep.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
activity_minutes = pd.Series({
    "Very Active": daily_activity["VeryActiveMinutes"].mean(),
    "Fairly Active": daily_activity["FairlyActiveMinutes"].mean(),
    "Lightly Active": daily_activity["LightlyActiveMinutes"].mean(),
    "Sedentary": daily_activity["SedentaryMinutes"].mean()
})

plt.figure(figsize=(8, 5))

plt.bar(
    activity_minutes.index,
    activity_minutes.values
)

plt.title("Average Daily Minutes by Activity Type")
plt.xlabel("Activity Type")
plt.ylabel("Average Minutes per Day")
plt.xticks(rotation=20)

plt.tight_layout()

chart_path = chart_folder / "average_minutes_by_activity_type.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)